In [12]:
# Instalar PyTorch con CUDA (primero, para evitar que sentence-transformers instale CPU)
%pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Core libs
%pip install pandas==2.2.2 matplotlib==3.9.0 \
datasets==2.20.0 pyarrow==15.0.2 \
lime==0.2.0.1 shap==0.45.1

  Using cached pandas-2.2.2.tar.gz (4.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [12 lines of output]
      + meson setup C:\Users\Usuario\AppData\Local\Temp\pip-install-5gf_tkup\pandas_397e3839907a4d43be4de92ea6b89d99 C:\Users\Usuario\AppData\Local\Temp\pip-install-5gf_tkup\pandas_397e3839907a4d43be4de92ea6b89d99\.mesonpy-p_tysn04\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\Usuario\AppData\Local\Temp\pip-install-5gf_tkup\pandas_397e3839907a4d43be4de92ea6b89d99\.mesonpy-p_tysn04\build\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.1
      Source dir: C:\Users\Usuario\AppData\Local\Temp\pip-install-5gf_tkup\pandas_397e3839907a4d43be4de92ea6b89d99
      Build dir: C:\Users\Usuario\AppData\Local\Temp\pip-install-5gf_tkup\pandas_397e3839907a4d43be4de92ea6b89d99\.mesonpy-p_tysn04\build
      Build type: native build
      Project name: pandas
      Project version: 

In [ ]:
import torch
import gc

from datasets import load_dataset
from datasets import Dataset

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
from transformers import AutoConfig

In [15]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [17]:
dataset = load_dataset("imdb")

In [18]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_MODEL_LENGTH = 2048
MAX_LENGTH = 2048

In [19]:
def build_prompt(review):

    prompt = f"""Review: {review}

Sentiment:"""

    return prompt


def build_full_text(review, label):

    sentiment = (
        "positive"
        if label == 1
        else "negative"
    )

    full_text = f"""Review: {review}

Sentiment: {sentiment}"""

    return full_text


def preprocess_dataset(dataset, tokenizer):

    input_ids_list = []
    attention_masks_list = []
    labels_list = []
    true_labels_list = []

    for example in dataset:

        review = example["text"]
        label = example["label"]

        prompt = build_prompt(review)
        full_text = build_full_text(review, label)

        # =========================
        # TOKENIZACIÓN
        # =========================

        full = tokenizer(
            full_text,
            add_special_tokens=False
        )

        input_ids = full["input_ids"]

        # attention mask inicial
        attention_mask = [1] * len(input_ids)

        # =========================
        # LABELS (SIN prompt_len)
        # =========================

        labels = []

        # reconstruimos prompt tokenizado directamente dentro del full_text
        prompt_text = build_prompt(review)

        prompt_ids = tokenizer(
            prompt_text,
            add_special_tokens=False
        )["input_ids"]

        # ⚠️ no asumimos alineación por slicing, buscamos corte seguro por longitud
        prompt_len = len(prompt_ids)

        for i, tok in enumerate(input_ids):

            if i < prompt_len:
                labels.append(-100)
            else:
                labels.append(tok)

        # =========================
        # PADDING / TRUNCATION UNIFORME
        # =========================

        if len(input_ids) > MAX_LENGTH:

            input_ids = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]

        else:

            pad_len = MAX_LENGTH - len(input_ids)

            input_ids += [tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labels += [-100] * pad_len

        # =========================
        # TRUE LABEL EXPLÍCITO
        # =========================

        true_label = "positive" if label == 1 else "negative"

        # =========================
        # STORE
        # =========================

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)
        true_labels_list.append(true_label)

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list,
        "true_label": true_labels_list
    })

In [20]:
# Model tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [21]:
def fits_model(example):

    text = build_full_text(
        example["text"],
        example["label"]
    )

    tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    return len(tokens) <= MAX_MODEL_LENGTH

In [22]:
# Reviews exceeding model context are removed from the dataset.
BATCH_SIZE = 1

filtered_test = dataset["test"].filter(
    fits_model
)

test_dataset = preprocess_dataset(filtered_test, tokenizer)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
        "true_label"
    ]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [23]:
teacher_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    attn_implementation="eager"
)

teacher_model.load_state_dict(
    torch.load(
        "./checkpoints/best_teacher_model.pt",
        map_location=device
    )
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11020.62it/s]


<All keys matched successfully>

# Métricas de comparación

In [ ]:
def frobenius_difference(
    x: torch.Tensor,
    y: torch.Tensor
):
    return torch.norm(
        x - y,
        p="fro"
    )


def cosine_similarity_tensor(
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = 1e-8
):
    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)

    similarity = F.cosine_similarity(
        x_flat.unsqueeze(0),
        y_flat.unsqueeze(0),
        dim=1,
        eps=eps
    )

    return similarity.squeeze()

def js_divergence_attention(
    attn_teacher: torch.Tensor,
    attn_student: torch.Tensor,
    eps: float = 1e-8
):
    # normalizar filas
    p = attn_teacher / (
        attn_teacher.sum(dim=-1, keepdim=True)
        + eps
    )

    q = attn_student / (
        attn_student.sum(dim=-1, keepdim=True)
        + eps
    )

    m = 0.5 * (p + q)

    kl_pm = torch.sum(
        p * torch.log(
            (p + eps) / (m + eps)
        ),
        dim=-1
    )

    kl_qm = torch.sum(
        q * torch.log(
            (q + eps) / (m + eps)
        ),
        dim=-1
    )

    jsd_rows = 0.5 * (
        kl_pm + kl_qm
    )

    return jsd_rows.mean()

# Rango efectivo

In [25]:
def effective_rank_participation_ratio(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    power = singular_values**2

    numerator = (
        power.sum()**2
    )

    denominator = (
        (power**2).sum()
        + eps
    )

    rank_eff = (
        numerator
        / denominator
    )

    return rank_eff


def effective_rank_entropy(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    p = singular_values / (
        singular_values.sum()
        + eps
    )

    entropy = -torch.sum(
        p * torch.log(p + eps)
    )

    rank_eff = torch.exp(entropy)

    return rank_eff

# Atención promedio

In [26]:
def compute_average_attentions(
    model,
    dataloader,
    device,
    max_batches=None,
    print_every=50
):

    model.eval()

    n_layers = (
        model.config.num_hidden_layers
    )

    attention_sums = [

        torch.zeros(
            (2048, 2048),
            dtype=torch.float32,
            device="cpu"
        )

        for _ in range(n_layers)
    ]

    n_examples = 0

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            dataloader
        ):

            if (
                max_batches is not None
                and batch_idx >= max_batches
            ):
                break

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_attentions=True
            )

            attentions = (
                outputs.attentions
            )

            batch_size = (
                input_ids.size(0)
            )

            for layer_idx in range(
                n_layers
            ):

                attn = attentions[
                    layer_idx
                ]

                # (B, H, S, S)
                # promedio heads
                attn = attn.mean(
                    dim=1
                )

                # (B, S, S)
                attn = (
                    attn
                    .float()
                    .cpu()
                )

                # suma batch
                attn_sum = attn.sum(
                    dim=0
                )

                attention_sums[
                    layer_idx
                ] += attn_sum

            n_examples += batch_size

            if (
                batch_idx
                % print_every
                == 0
            ):

                print(
                    f"Batch "
                    f"{batch_idx} | "
                    f"Examples "
                    f"{n_examples}"
                )

            del (
                input_ids,
                attention_mask,
                outputs,
                attentions,
                attn
            )

            gc.collect()

            torch.cuda.empty_cache()

    attention_means = [

        attn_sum / n_examples

        for attn_sum
        in attention_sums
    ]

    return attention_means

## Teacher

In [27]:
teacher_model.to(device)
teacher_model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [28]:
teacher_model.config.output_attentions = True

In [29]:
teacher_attentions = compute_average_attentions(teacher_model, test_loader, device)

Batch 0 | Examples 1
Batch 50 | Examples 51
Batch 100 | Examples 101
Batch 150 | Examples 151
Batch 200 | Examples 201
Batch 250 | Examples 251
Batch 300 | Examples 301
Batch 350 | Examples 351
Batch 400 | Examples 401
Batch 450 | Examples 451
Batch 500 | Examples 501
Batch 550 | Examples 551
Batch 600 | Examples 601
Batch 650 | Examples 651
Batch 700 | Examples 701
Batch 750 | Examples 751
Batch 800 | Examples 801
Batch 850 | Examples 851
Batch 900 | Examples 901
Batch 950 | Examples 951
Batch 1000 | Examples 1001
Batch 1050 | Examples 1051
Batch 1100 | Examples 1101
Batch 1150 | Examples 1151
Batch 1200 | Examples 1201
Batch 1250 | Examples 1251
Batch 1300 | Examples 1301
Batch 1350 | Examples 1351
Batch 1400 | Examples 1401
Batch 1450 | Examples 1451
Batch 1500 | Examples 1501
Batch 1550 | Examples 1551
Batch 1600 | Examples 1601
Batch 1650 | Examples 1651
Batch 1700 | Examples 1701
Batch 1750 | Examples 1751
Batch 1800 | Examples 1801
Batch 1850 | Examples 1851
Batch 1900 | Example

In [30]:
teacher_model.cpu()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [31]:
teacher_avg_attentions = [
    (teacher_attentions[2*i] + teacher_attentions[2*i + 1]) / 2
    for i in range(11)
]

In [32]:
def rollout_pair(A1, A2):
    I = torch.eye(A1.shape[-1], device=A1.device)
    
    # agregar residual y renormalizar
    A1 = A1 + I
    A1 = A1 / A1.sum(dim=-1, keepdim=True)
    
    A2 = A2 + I
    A2 = A2 / A2.sum(dim=-1, keepdim=True)
    
    return A1 @ A2

teacher_rollouts = [rollout_pair(teacher_attentions[2*i], teacher_attentions[2*i+1]) for i in range(11)]

## Teacher - Student comparison

In [33]:
# cargar config original
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)

# student de 11 capas
config.num_hidden_layers = 11

# crear arquitectura vacía
student_model = (
    AutoModelForCausalLM.from_config(
        config,
        attn_implementation="eager"
    )
)

# cargar pesos
student_model.load_state_dict(
    torch.load(
        "./checkpoints/best_student_model.pt",
        map_location=device
    )
)

student_model.to(device)
student_model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-10): 11 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [34]:
student_model.config.output_attentions = True

In [35]:
student_attentions = compute_average_attentions(student_model, test_loader, device)

Batch 0 | Examples 1
Batch 50 | Examples 51
Batch 100 | Examples 101
Batch 150 | Examples 151
Batch 200 | Examples 201
Batch 250 | Examples 251
Batch 300 | Examples 301
Batch 350 | Examples 351
Batch 400 | Examples 401
Batch 450 | Examples 451
Batch 500 | Examples 501
Batch 550 | Examples 551
Batch 600 | Examples 601
Batch 650 | Examples 651
Batch 700 | Examples 701
Batch 750 | Examples 751
Batch 800 | Examples 801
Batch 850 | Examples 851
Batch 900 | Examples 901
Batch 950 | Examples 951
Batch 1000 | Examples 1001
Batch 1050 | Examples 1051
Batch 1100 | Examples 1101
Batch 1150 | Examples 1151
Batch 1200 | Examples 1201
Batch 1250 | Examples 1251
Batch 1300 | Examples 1301
Batch 1350 | Examples 1351
Batch 1400 | Examples 1401
Batch 1450 | Examples 1451
Batch 1500 | Examples 1501
Batch 1550 | Examples 1551
Batch 1600 | Examples 1601
Batch 1650 | Examples 1651
Batch 1700 | Examples 1701
Batch 1750 | Examples 1751
Batch 1800 | Examples 1801
Batch 1850 | Examples 1851
Batch 1900 | Example

In [36]:
student_model.cpu()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-10): 11 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [37]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9271)
tensor(0.2688)
tensor(0.3348)
tensor(0.1360)
tensor(0.1623)
tensor(0.1579)
tensor(0.1731)
tensor(0.1687)
tensor(0.2197)
tensor(0.2192)
tensor(0.3066)


Similitud coseno con Attention Rollout
tensor(0.2752)
tensor(0.2268)
tensor(0.3322)
tensor(0.1484)
tensor(0.1930)
tensor(0.1914)
tensor(0.2138)
tensor(0.2225)
tensor(0.2439)
tensor(0.2334)
tensor(0.3291)


In [38]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.3432)
tensor(18.9485)
tensor(35.8680)
tensor(39.6938)
tensor(25.9925)
tensor(22.6046)
tensor(18.0666)
tensor(17.6782)
tensor(19.0674)
tensor(19.9193)
tensor(20.4072)


Distancia Frobenius con Attention Rollout
tensor(11.3760)
tensor(21.6454)
tensor(30.4318)
tensor(32.8054)
tensor(25.0566)
tensor(23.0084)
tensor(19.9416)
tensor(19.8836)
tensor(20.7777)
tensor(21.3191)
tensor(21.4697)


In [ ]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(js_divergence_attention(sattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0122)
tensor(0.1646)
tensor(0.3903)
tensor(0.4936)
tensor(0.2674)
tensor(0.2390)
tensor(0.1921)
tensor(0.1894)
tensor(0.2114)
tensor(0.2218)
tensor(0.2148)


JS promediada por fila con Attention Rollout
tensor(0.1108)
tensor(0.2943)
tensor(0.4644)
tensor(0.5489)
tensor(0.3641)
tensor(0.3247)
tensor(0.2791)
tensor(0.2791)
tensor(0.3001)
tensor(0.3124)
tensor(0.3020)


In [41]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(sattn)} | {effective_rank_participation_ratio(tattn)} | {effective_rank_participation_ratio(tattn2)}")


Participation ratio
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 2.435163736343384 | 2.976497173309326 | 365.10394287109375
1 | 2.089698553085327 | 1.0099334716796875 | 1.8481037616729736
2 | 1.8754035234451294 | 1.00034499168396 | 1.3134498596191406
3 | 2.092130422592163 | 1.0005322694778442 | 1.2850630283355713
4 | 4.12630558013916 | 1.0030649900436401 | 1.5619888305664062
5 | 8.57667064666748 | 1.0039870738983154 | 1.7202398777008057
6 | 9.733598709106445 | 1.0108572244644165 | 2.138787269592285
7 | 10.104803085327148 | 1.0122770071029663 | 2.1546192169189453
8 | 4.953476428985596 | 1.0085184574127197 | 1.9759875535964966
9 | 3.4227583408355713 | 1.0054681301116943 | 1.9005166292190552
10 | 3.2704408168792725 | 1.0128203630447388 | 1.8738715648651123


In [42]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(sattn)} | {effective_rank_entropy(tattn)} | {effective_rank_entropy(tattn2)}")

Entropy based effective rank
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 77.74598693847656 | 257.76190185546875 | 2026.011474609375
1 | 75.56600189208984 | 32.70494842529297 | 1817.4815673828125
2 | 137.7758026123047 | 4.429290294647217 | 1666.9998779296875
3 | 159.863037109375 | 5.763463497161865 | 1648.0753173828125
4 | 354.9866943359375 | 15.971895217895508 | 1764.5404052734375
5 | 534.8629760742188 | 20.12454605102539 | 1798.460205078125
6 | 400.9078063964844 | 50.65095138549805 | 1849.495849609375
7 | 419.5435791015625 | 64.90833282470703 | 1850.6656494140625
8 | 348.2325439453125 | 49.009342193603516 | 1833.6090087890625
9 | 298.16217041015625 | 34.75936508178711 | 1825.2310791015625
10 | 382.67132568359375 | 100.71648406982422 | 1819.1229248046875


## Teacher - Student local comparison

In [43]:
# crear arquitectura vacía
student_local_model = (
    AutoModelForCausalLM.from_config(
        config,
        attn_implementation="eager"
    )
)

# cargar pesos
student_local_model.load_state_dict(
    torch.load(
        "./checkpoints/best_student_local_model.pt",
        map_location=device
    )
)

student_local_model.to(device)
student_local_model.eval()

[transformers] The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-10): 11 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [44]:
student_local_model.config.output_attentions = True

In [45]:
student_local_attentions = compute_average_attentions(student_local_model, test_loader, device)

Batch 0 | Examples 1
Batch 50 | Examples 51
Batch 100 | Examples 101
Batch 150 | Examples 151
Batch 200 | Examples 201
Batch 250 | Examples 251
Batch 300 | Examples 301
Batch 350 | Examples 351
Batch 400 | Examples 401
Batch 450 | Examples 451
Batch 500 | Examples 501
Batch 550 | Examples 551
Batch 600 | Examples 601
Batch 650 | Examples 651
Batch 700 | Examples 701
Batch 750 | Examples 751
Batch 800 | Examples 801
Batch 850 | Examples 851
Batch 900 | Examples 901
Batch 950 | Examples 951
Batch 1000 | Examples 1001
Batch 1050 | Examples 1051
Batch 1100 | Examples 1101
Batch 1150 | Examples 1151
Batch 1200 | Examples 1201
Batch 1250 | Examples 1251
Batch 1300 | Examples 1301
Batch 1350 | Examples 1351
Batch 1400 | Examples 1401
Batch 1450 | Examples 1451
Batch 1500 | Examples 1501
Batch 1550 | Examples 1551
Batch 1600 | Examples 1601
Batch 1650 | Examples 1651
Batch 1700 | Examples 1701
Batch 1750 | Examples 1751
Batch 1800 | Examples 1801
Batch 1850 | Examples 1851
Batch 1900 | Example

In [46]:
student_local_model.cpu()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-10): 11 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [47]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9530)
tensor(0.3106)
tensor(0.5676)
tensor(0.3265)
tensor(0.2454)
tensor(0.3558)
tensor(0.2092)
tensor(0.1995)
tensor(0.2492)
tensor(0.1810)
tensor(0.2851)


Similitud coseno con Attention Rollout
tensor(0.2877)
tensor(0.2588)
tensor(0.5316)
tensor(0.3308)
tensor(0.2537)
tensor(0.3669)
tensor(0.2390)
tensor(0.2619)
tensor(0.2897)
tensor(0.2456)
tensor(0.3415)


In [48]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.0846)
tensor(18.7224)
tensor(33.7373)
tensor(38.8643)
tensor(25.6862)
tensor(21.5705)
tensor(17.9186)
tensor(17.5799)
tensor(18.9334)
tensor(20.0786)
tensor(20.4738)


Distancia Frobenius con Attention Rollout
tensor(11.3316)
tensor(21.4776)
tensor(28.6160)
tensor(32.0087)
tensor(24.8451)
tensor(22.1283)
tensor(19.8215)
tensor(19.6842)
tensor(20.5559)
tensor(21.2283)
tensor(21.3467)


In [ ]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(js_divergence_attention(sattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0086)
tensor(0.1597)
tensor(0.3406)
tensor(0.4627)
tensor(0.2544)
tensor(0.2040)
tensor(0.1812)
tensor(0.1827)
tensor(0.2033)
tensor(0.2221)
tensor(0.2092)


JS promediada por fila con Attention Rollout
tensor(0.1042)
tensor(0.2893)
tensor(0.4227)
tensor(0.5173)
tensor(0.3564)
tensor(0.2960)
tensor(0.2689)
tensor(0.2678)
tensor(0.2884)
tensor(0.3035)
tensor(0.2924)


In [53]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_local_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(sattn)} | {effective_rank_participation_ratio(tattn)} | {effective_rank_participation_ratio(tattn2)}")


Participation ratio
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 2.502690553665161 | 2.976497173309326 | 365.10394287109375
1 | 1.7893288135528564 | 1.0099334716796875 | 1.8481037616729736
2 | 1.2827714681625366 | 1.00034499168396 | 1.3134498596191406
3 | 2.1537716388702393 | 1.0005322694778442 | 1.2850630283355713
4 | 3.278672695159912 | 1.0030649900436401 | 1.5619888305664062
5 | 4.7921061515808105 | 1.0039870738983154 | 1.7202398777008057
6 | 12.610973358154297 | 1.0108572244644165 | 2.138787269592285
7 | 15.858839988708496 | 1.0122770071029663 | 2.1546192169189453
8 | 7.0641045570373535 | 1.0085184574127197 | 1.9759875535964966
9 | 8.69315242767334 | 1.0054681301116943 | 1.9005166292190552
10 | 8.683012008666992 | 1.0128203630447388 | 1.8738715648651123


In [54]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (sattn, tattn, tattn2) in enumerate(zip(student_local_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(sattn)} | {effective_rank_entropy(tattn)} | {effective_rank_entropy(tattn2)}")

Entropy based effective rank
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 93.27125549316406 | 257.76190185546875 | 2026.011474609375
1 | 56.27619171142578 | 32.70494842529297 | 1817.4815673828125
2 | 17.545154571533203 | 4.429290294647217 | 1666.9998779296875
3 | 207.46107482910156 | 5.763463497161865 | 1648.0753173828125
4 | 259.7679443359375 | 15.971895217895508 | 1764.5404052734375
5 | 396.5259094238281 | 20.12454605102539 | 1798.460205078125
6 | 495.88458251953125 | 50.65095138549805 | 1849.495849609375
7 | 457.3091125488281 | 64.90833282470703 | 1850.6656494140625
8 | 414.09893798828125 | 49.009342193603516 | 1833.6090087890625
9 | 428.9598083496094 | 34.75936508178711 | 1825.2310791015625
10 | 466.45416259765625 | 100.71648406982422 | 1819.1229248046875


# Student - Student comparison

In [ ]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(cosine_similarity_tensor(sattn, slattn))

Similitud coseno con promedio de atenciones
tensor(0.9713)
tensor(0.9258)
tensor(0.8185)
tensor(0.8940)
tensor(0.9451)
tensor(0.9049)
tensor(0.9586)
tensor(0.9706)
tensor(0.9590)
tensor(0.8860)
tensor(0.8624)


In [ ]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(frobenius_difference(sattn, slattn))

Distancia Frobenius con promedio de atenciones
tensor(0.8196)
tensor(1.8418)
tensor(4.2136)
tensor(1.9827)
tensor(1.3581)
tensor(2.6141)
tensor(1.4082)
tensor(1.3172)
tensor(1.2321)
tensor(2.0819)
tensor(2.4813)


In [ ]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(js_divergence_attention(sattn, slattn))

JS promediada por fila con promedio de atenciones


tensor(0.0055)
tensor(0.0100)
tensor(0.0327)
tensor(0.0304)
tensor(0.0128)
tensor(0.0225)
tensor(0.0150)
tensor(0.0128)
tensor(0.0126)
tensor(0.0185)
tensor(0.0384)


In [67]:
# Participation ratio
print("Participation ratio")
print("index | student | student_local")
print("-----------------------------------------------")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(f"{i} | {effective_rank_participation_ratio(sattn)} | {effective_rank_participation_ratio(slattn)}")


Participation ratio
index | student | student_local
-----------------------------------------------
0 | 2.435163736343384 | 2.502690553665161
1 | 2.089698553085327 | 1.7893288135528564
2 | 1.8754035234451294 | 1.2827714681625366
3 | 2.092130422592163 | 2.1537716388702393
4 | 4.12630558013916 | 3.278672695159912
5 | 8.57667064666748 | 4.7921061515808105
6 | 9.733598709106445 | 12.610973358154297
7 | 10.104803085327148 | 15.858839988708496
8 | 4.953476428985596 | 7.0641045570373535
9 | 3.4227583408355713 | 8.69315242767334
10 | 3.2704408168792725 | 8.683012008666992


In [68]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | student_local")
print("-----------------------------------------------")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(f"{i} | {effective_rank_entropy(sattn)} | {effective_rank_entropy(slattn)} ")

Entropy based effective rank
index | student | student_local
-----------------------------------------------
0 | 77.74598693847656 | 93.27125549316406 
1 | 75.56600189208984 | 56.27619171142578 
2 | 137.7758026123047 | 17.545154571533203 
3 | 159.863037109375 | 207.46107482910156 
4 | 354.9866943359375 | 259.7679443359375 
5 | 534.8629760742188 | 396.5259094238281 
6 | 400.9078063964844 | 495.88458251953125 
7 | 419.5435791015625 | 457.3091125488281 
8 | 348.2325439453125 | 414.09893798828125 
9 | 298.16217041015625 | 428.9598083496094 
10 | 382.67132568359375 | 466.45416259765625 


# Comparación con promedio y rollout total

In [72]:
teacher_total_avg_attention = sum(teacher_attentions)/len(teacher_attentions)

def total_rollout(attentions):
    I = torch.eye(2048, device="cpu")
    R = None
    for A in attentions:

        attn = A + I
        attn = attn / attn.sum(dim=-1, keepdim=True)

        if R is None:
            R = attn
        else:
            R = R @ attn
    
    return R

teacher_total_rollout = total_rollout(teacher_attentions)

In [75]:
student_total_avg_attention = sum(student_attentions)/len(student_attentions)
student_total_rollout = total_rollout(student_attentions)

In [76]:
student_local_total_avg_attention = sum(student_local_attentions)/len(student_local_attentions)
student_local_total_rollout = total_rollout(student_local_attentions)

In [79]:
# Cosine similarity entre attention maps
print("Similitud coseno")

print("Promedio student - Promedio teacher")
print(cosine_similarity_tensor(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(cosine_similarity_tensor(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(cosine_similarity_tensor(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(cosine_similarity_tensor(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(cosine_similarity_tensor(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(cosine_similarity_tensor(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(cosine_similarity_tensor(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(cosine_similarity_tensor(student_total_rollout, student_local_total_rollout))

Similitud coseno
Promedio student - Promedio teacher
tensor(0.2492)
Promedio student - Rollout teacher
tensor(0.1731)
Promedio student local - Promedio teacher
tensor(0.3478)
Promedio student local - Rollout teacher
tensor(0.2693)
Rollout student - Promedio teacher
tensor(0.9534)
Rollout student - Rollout teacher
tensor(0.9434)
Rollout student local - Promedio teacher
tensor(0.9534)
Rollout student local - Rollout teacher
tensor(0.9434)
Promedio student - Promedio student local
tensor(0.9865)
Promedio student - Rollout student local
tensor(0.2977)
Rollout student - Promedio student local
tensor(0.4221)
Rollout student - Rollout student local
tensor(0.9953)


In [ ]:
# Distancia Frobenius entre attention maps
print("Distancia Frobenius")

print("Promedio student - Promedio teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(frobenius_difference(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(frobenius_difference(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(frobenius_difference(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(frobenius_difference(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(frobenius_difference(student_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(frobenius_difference(student_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(frobenius_difference(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(frobenius_difference(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(frobenius_difference(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(frobenius_difference(student_total_rollout, student_local_total_rollout))

Similitud coseno
Promedio student - Promedio teacher
tensor(21.4037)
Promedio student - Rollout teacher
tensor(44.7235)
Promedio student local - Promedio teacher
tensor(20.9771)
Promedio student local - Rollout teacher
tensor(44.3146)
Rollout student - Promedio teacher
tensor(6.8544)
Rollout student - Rollout teacher
tensor(27.6911)
Rollout student local - Promedio teacher
tensor(6.8544)
Rollout student local - Rollout teacher
tensor(27.6911)
Promedio student - Promedio student local
tensor(0.6835)
Promedio student - Rollout student local
tensor(21.2566)
Rollout student - Promedio student local
tensor(18.0665)
Rollout student - Rollout student local
tensor(3.3768)


In [ ]:
# JS local entre attention maps
print("JS local")

print("Promedio student - Promedio teacher")
print(js_divergence_attention(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(js_divergence_attention(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(js_divergence_attention(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(js_divergence_attention(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(js_divergence_attention(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(js_divergence_attention(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(js_divergence_attention(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(js_divergence_attention(student_total_rollout, student_local_total_rollout))

JS local
Promedio student - Promedio teacher
tensor(0.1946)
Promedio student - Rollout teacher
tensor(0.6554)
Promedio student local - Promedio teacher
tensor(0.1815)
Promedio student local - Rollout teacher
tensor(0.6374)
Rollout student - Promedio teacher
tensor(0.1787)
Rollout student - Rollout teacher
tensor(0.2719)
Rollout student local - Promedio teacher
tensor(0.1787)
Rollout student local - Rollout teacher
tensor(0.2719)
Promedio student - Promedio student local
tensor(0.0024)
Promedio student - Rollout student local
tensor(0.3120)
Rollout student - Promedio student local
tensor(0.2879)
Rollout student - Rollout student local
tensor(0.0030)


In [83]:
# Participation ratio
print("Participation ratio")
print("Teacher avg")
print(effective_rank_participation_ratio(teacher_total_avg_attention))
print("Student avg")
print(effective_rank_participation_ratio(student_total_avg_attention))
print("Student local avg")
print(effective_rank_participation_ratio(student_local_total_avg_attention))
print("Teacher rollout")
print(effective_rank_participation_ratio(teacher_total_rollout))
print("Student rollout")
print(effective_rank_participation_ratio(student_total_rollout))
print("Student local rollout")
print(effective_rank_participation_ratio(student_local_total_rollout))

Participation ratio
Teacher avg
tensor(1.0052)
Student avg
tensor(3.2826)
Student local avg
tensor(3.2873)
Teacher rollout
tensor(1.)
Student rollout
tensor(1.0055)
Student local rollout
tensor(1.0022)


In [84]:
# Entropy based effective rank
print("Entropy based effective rank")
print("Teacher avg")
print(effective_rank_entropy(teacher_total_avg_attention))
print("Student avg")
print(effective_rank_entropy(student_total_avg_attention))
print("Student local avg")
print(effective_rank_entropy(student_local_total_avg_attention))
print("Teacher rollout")
print(effective_rank_entropy(teacher_total_rollout))
print("Student rollout")
print(effective_rank_entropy(student_total_rollout))
print("Student local rollout")
print(effective_rank_entropy(student_local_total_rollout))

Entropy based effective rank
Teacher avg
tensor(31.1072)
Student avg
tensor(336.6037)
Student local avg
tensor(362.2327)
Teacher rollout
tensor(1.0003)
Student rollout
tensor(2.4227)
Student local rollout
tensor(2.1826)
